<a href="https://colab.research.google.com/github/shreyaghora/ArrhythmiaHub/blob/main/Code/Smote_Wilcoxon_Signed_Rank_Test_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm
!pip install catboost
!pip install xgboost
!pip install specificity
!pip install imbalanced-learn
!pip install lime
!pip install xlsxwriter

!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=9f0df531d681964f52a145fd3f8be094ff5e62fbc50943baede21a7adca9be84
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.2 MB/s eta 0:00:00
time: 342 µs (started: 2026-05-19 08:03:24 +00:00)


In [ ]:
import numpy as np
import pandas as pd
import time
import shap
import matplotlib.pyplot as plt

from scipy.stats import uniform
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate

from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix, balanced_accuracy_score

time: 9.75 s (started: 2026-05-19 08:21:53 +00:00)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
time: 25 s (started: 2026-05-19 08:22:30 +00:00)


In [ ]:
df = pd.read_csv("INCART 2-lead Arrhythmia Database.csv")
df.head(50)

,record,type,0_pre-RR,0_post-RR,0_pPeak,0_tPeak,0_rPeak,0_sPeak,0_qPeak,0_qrs_interval,...,1_qPeak,1_qrs_interval,1_pq_interval,1_qt_interval,1_st_interval,1_qrs_morph0,1_qrs_morph1,1_qrs_morph2,1_qrs_morph3,1_qrs_morph4
0,I01,N,163,165,0.069610,-0.083281,0.614133,-0.392761,0.047159,15,...,-0.023370,14,3,23,6,-0.023370,-0.011650,0.082608,0.101373,-0.183387
1,I01,N,165,166,-0.097030,0.597254,-0.078704,-0.078704,-0.137781,3,...,0.081637,15,5,27,7,0.081637,0.102992,0.191225,0.217544,-0.068248
2,I01,N,166,102,0.109399,0.680528,-0.010649,-0.010649,-0.720620,6,...,-0.148539,33,13,52,6,-0.148539,-0.060620,0.081080,0.204400,0.335172
3,I01,VEB,102,231,0.176376,0.256431,-0.101098,-0.707525,-0.101098,4,...,0.046898,21,9,34,4,0.046898,0.083728,0.279512,0.526785,0.450969
4,I01,N,231,165,0.585577,0.607461,-0.083499,-0.083499,-0.167858,3,...,-0.112552,32,5,43,6,-0.112552,0.012989,0.091491,0.134004,0.265232
5,I01,N,165,163,0.019637,-0.111671,0.613374,-0.416937,-0.108554,21,...,-0.088215,32,8,47,7,-0.088215,-0.001563,0.082968,0.136342,0.308107
6,I01,N,163,166,-0.156761,0.528148,-0.145137,-0.145137,-0.183749,2,...,0.066682,16,5,27,6,0.066682,0.084804,0.174772,0.224360,-0.011853
7,I01,N,166,161,-0.003887,0.216949,1.120258,0.164411,-0.725340,33,...,-0.198099,3,4,14,7,-0.198099,-0.198099,-0.322683,-0.322683,-0.429821
8,I01,N,161,160,-0.006721,0.164998,0.601190,-0.307812,-0.035103,15,...,0.062813,14,9,29,6,0.062813,0.080551,0.159296,0.108512,-0.248600
9,I01,N,160,158,-0.187870,0.544414,-0.018564,-0.018564,-0.194883,5,...,0.127967,14,5,26,7,0.127967,0.147617,0.247298,0.264875,-0.032502


time: 1.25 s (started: 2026-05-19 08:22:58 +00:00)


In [ ]:
# # 3. Drop Feature
df.drop('record', axis=1, inplace=True)

df = df[df['type'] != 'Q']

# 5. Encode Target
df['type'] = df['type'].map({'F': 0, 'N': 1, 'SVEB': 2, 'VEB': 3});
X = df.drop('type', axis=1)
y = df['type']

feature_names = X.columns;

# 9. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. Apply SMOTE
sm = SMOTE(random_state=42, k_neighbors=2)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

# 8. SCALE DATA
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)


time: 2.49 s (started: 2026-05-19 08:23:00 +00:00)


In [ ]:
def hyper_parameter_tune(model, param_dist, X_train, y_train):

    search = RandomizedSearchCV(
          model,
          param_distributions=param_dist,
          n_iter=5,
          cv=5,
          refit='roc_auc',
          n_jobs=-1,
          random_state=42,
          return_train_score=False
          )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    print("Best parameters:", search.best_params_)
    print("Best CV accuracy: {:.4f}".format(search.best_score_))


    return best_model

time: 658 µs (started: 2026-05-19 08:23:02 +00:00)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    balanced_accuracy_score
)

from sklearn.model_selection import KFold
from sklearn.preprocessing import label_binarize

import numpy as np
import pandas as pd
import time


def evaluate_models(name, model, X, y):

    result = []

    print(f"\nTrain Model: {name}\n")

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    classes = np.unique(y)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Train
        start_time = time.time()

        model.fit(X_train, y_train)

        training_time = time.time() - start_time

        # Predict
        y_pred = model.predict(X_test)

        # Metrics
        acc = accuracy_score(y_test, y_pred)

        prec = precision_score(y_test,y_pred,average='weighted',zero_division=0)

        rec = recall_score(y_test,y_pred,average='weighted',zero_division=0)

        f1 = f1_score(y_test,y_pred,average='weighted',zero_division=0)

        balanced_acc = balanced_accuracy_score(y_test, y_pred)

        mcc = matthews_corrcoef(y_test, y_pred)

        kappa = cohen_kappa_score(y_test, y_pred)

        try:
            y_prob = model.predict_proba(X_test)

            y_test_bin = label_binarize(y_test, classes=classes)

            auc = roc_auc_score(
                y_test_bin,
                y_prob,
                multi_class='ovr'
            )

        except:
            auc = np.nan

        result.append({
            "Classifier": name,
            "Fold": fold + 1,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "AUC": auc,
            "MCC": mcc,
            "Kappa": kappa,
            "Balanced Accuracy": balanced_acc,
            "Training Time (s)": training_time
        })

    return pd.DataFrame(result)

time: 2.24 ms (started: 2026-05-19 08:23:04 +00:00)


In [ ]:
#  SHAP EXPLAINABILITY
def shap_explain(model, explainer_type, X_test_scaled, feature_names, X_test = X_test):
  print("\n--- SHAP EXPLANATION ---")


  # Use TreeExplainer (since RandomForest is tree-based)
  explainer = explainer_type(model)

  # Compute SHAP values
  shap_values = explainer.shap_values(X_test_scaled)

  # -------------------------------
  # 1. GLOBAL FEATURE IMPORTANCE
  # -------------------------------
  plt.figure()
  shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_names, plot_type="bar")

  # -------------------------------
  # 2. DETAILED SUMMARY PLOT
  # -------------------------------
  plt.figure()

  shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_names)
  plt.figure()

  # # -------------------------------
  # # 3. DEPENDENCE PLOT
  # # -------------------------------
  # # Change feature name as needed
  # shap.dependence_plot(
  #     feature_names[0],  # example feature
  #     shap_values,
  #     X_test_scaled,
  #     feature_names=feature_names # Provide feature names for clarity
  # )

  # # -------------------------------
  # # 4. LOCAL EXPLANATION (FORCE PLOT)
  # # -------------------------------
  # # Use expected value of class 1
  # expected_value = explainer.expected_value

  # plt.figure()
  # shap.force_plot(
  #     expected_value,
  #     shap_values[0], # shap_vals is now (num_samples, num_features), so shap_vals[0] is for the first sample
  #     X_test.iloc[0],
  #     matplotlib=True
  # )

time: 945 µs (started: 2026-05-19 08:23:08 +00:00)


In [ ]:
# LIME EXPLAINABILITY

def lime_explain(model, X_train_scaled, X_test_scaled, feature_names, num_samples=100):

    print("\n--- LIME EXPLANATION ---")

    from lime.lime_tabular import LimeTabularExplainer
    import numpy as np
    import pandas as pd

    X_train_np = X_train_scaled.to_numpy() if hasattr(X_train_scaled, "to_numpy") else X_train_scaled
    X_test_np = X_test_scaled.to_numpy() if hasattr(X_test_scaled, "to_numpy") else X_test_scaled


    lime_explainer = LimeTabularExplainer(
        training_data=X_train_np,
        feature_names=feature_names,
        class_names=model.classes_.astype(str) if hasattr(model, "classes_") else None,
        mode='classification'
    )


    # 1. LOCAL EXPLANATION

    print("\n--- LOCAL EXPLANATION ---")

    lime_exp = lime_explainer.explain_instance(
        X_test_np[0],
        model.predict_proba
    )

    lime_exp.show_in_notebook(show_table=True)

    # 2. GLOBAL EXPLANATION

    print("\n--- GLOBAL EXPLANATION (approx via sampling) ---")

    feature_importance = np.zeros(X_train_np.shape[1])


    indices = np.random.choice(len(X_test_np), min(num_samples, len(X_test_np)), replace=False)

    for i in indices:
        exp = lime_explainer.explain_instance(
            X_test_np[i],
            model.predict_proba,
            num_features=len(feature_names)
        )

        for feature, weight in exp.as_list():

            feature_index = next(
                (j for j, name in enumerate(feature_names) if name in feature),
                None
            )

            if feature_index is not None:
                feature_importance[feature_index] += abs(weight)

    # Normalize importance
    feature_importance /= len(indices)

    # Create dataframe
    global_lime = pd.DataFrame({
        "feature": feature_names,
        "importance": feature_importance
    }).sort_values(by="importance", ascending=False)

    print(global_lime)


time: 4.14 ms (started: 2026-05-19 08:23:11 +00:00)


In [ ]:
from sklearn.neural_network import MLPClassifier

name="MLP"

#Model
model =   MLPClassifier(max_iter=300)
param_dist = {
      "hidden_layer_sizes": [(64,64), (128,64)]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)

MLP = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
MLP

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=5. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'hidden_layer_sizes': (128, 64)}
Best CV accuracy: 0.9989

Train Model: MLP



,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,MLP,1,0.999349,0.999349,0.999349,0.999348,0.999953,0.999131,0.999131,0.999347,351.834801
1,MLP,2,0.998941,0.998942,0.998941,0.998941,0.999968,0.998589,0.998588,0.998939,403.945937
2,MLP,3,0.998269,0.998272,0.998269,0.998269,0.999913,0.997694,0.997693,0.998279,582.365937
3,MLP,4,0.997944,0.997952,0.997944,0.997944,0.999955,0.997261,0.997258,0.997950,442.710343
4,MLP,5,0.999226,0.999227,0.999226,0.999226,0.999947,0.998969,0.998968,0.999222,368.395176
5,MLP,6,0.998921,0.998921,0.998921,0.998921,0.999960,0.998561,0.998561,0.998916,443.672083
6,MLP,7,0.999104,0.999105,0.999104,0.999104,0.999937,0.998806,0.998806,0.999108,329.817914
7,MLP,8,0.999491,0.999491,0.999491,0.999491,0.999967,0.999321,0.999321,0.999487,379.938530
8,MLP,9,0.999328,0.999328,0.999328,0.999328,0.999983,0.999104,0.999104,0.999331,352.115982
9,MLP,10,0.998371,0.998373,0.998371,0.998371,0.999976,0.997829,0.997828,0.998375,313.163197


time: 1h 44min 4s (started: 2026-05-16 22:07:16 +00:00)


time: 1h 44min 4s (started: 2026-05-16 22:07:16 +00:00)


time: 1h 44min 4s (started: 2026-05-16 22:07:16 +00:00)


In [ ]:
from xgboost import XGBClassifier

name ="XGBoost"

#Model
model =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')

param_dist = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
            "max_depth": [3, 6]
}
best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)

XGBoost = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
XGBoost

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:03:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}
Best CV accuracy: 0.9988

Train Model: XGBoost



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:04:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:05:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:06:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:07:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,XGBoost,1,0.998717,0.998719,0.998717,0.998717,0.999969,0.998291,0.998290,0.998713,53.321095
1,XGBoost,2,0.998901,0.998901,0.998901,0.998900,0.999973,0.998535,0.998534,0.998898,53.340144
2,XGBoost,3,0.998962,0.998962,0.998962,0.998961,0.999960,0.998616,0.998616,0.998968,53.106362
3,XGBoost,4,0.998840,0.998840,0.998840,0.998839,0.999978,0.998453,0.998453,0.998845,50.341409
4,XGBoost,5,0.998840,0.998840,0.998840,0.998839,0.999970,0.998453,0.998453,0.998834,52.192216
5,XGBoost,6,0.998717,0.998718,0.998717,0.998717,0.999986,0.998290,0.998290,0.998710,53.038486
6,XGBoost,7,0.998534,0.998536,0.998534,0.998534,0.999972,0.998046,0.998045,0.998541,52.850798
7,XGBoost,8,0.998982,0.998982,0.998982,0.998982,0.999990,0.998643,0.998643,0.998974,53.357474
8,XGBoost,9,0.999063,0.999064,0.999063,0.999063,0.999990,0.998752,0.998751,0.999067,53.352210
9,XGBoost,10,0.999226,0.999226,0.999226,0.999226,0.999992,0.998969,0.998968,0.999227,51.083619


time: 22min 10s (started: 2026-05-16 23:51:21 +00:00)


time: 22min 10s (started: 2026-05-16 23:51:21 +00:00)


time: 22min 10s (started: 2026-05-16 23:51:21 +00:00)


In [ ]:
from sklearn.tree import DecisionTreeClassifier


name="DecisionTree"

#Model
model = DecisionTreeClassifier()

# Search space
param_dist = {

    "max_depth":[5,10,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)

DecisionTree = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
DecisionTree

Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': None}
Best CV accuracy: 0.9958

Train Model: DecisionTree



,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,DecisionTree,1,0.995093,0.995090,0.995093,0.995090,0.997642,0.993459,0.993458,0.995082,50.872642
1,DecisionTree,2,0.996254,0.996254,0.996254,0.996253,0.998207,0.995006,0.995005,0.996250,48.959619
2,DecisionTree,3,0.996254,0.996253,0.996254,0.996253,0.998354,0.995006,0.995005,0.996265,50.765121
3,DecisionTree,4,0.995582,0.995582,0.995582,0.995581,0.998009,0.994110,0.994109,0.995591,49.411748
4,DecisionTree,5,0.996111,0.996111,0.996111,0.996110,0.998194,0.994816,0.994815,0.996102,50.692788
5,DecisionTree,6,0.996050,0.996049,0.996050,0.996049,0.998193,0.994734,0.994734,0.996048,50.443776
6,DecisionTree,7,0.996010,0.996008,0.996010,0.996008,0.998017,0.994680,0.994679,0.996026,50.064588
7,DecisionTree,8,0.996172,0.996172,0.996172,0.996172,0.998133,0.994897,0.994896,0.996158,50.778782
8,DecisionTree,9,0.995623,0.995622,0.995623,0.995622,0.997926,0.994164,0.994164,0.995630,49.824689
9,DecisionTree,10,0.996641,0.996641,0.996641,0.996640,0.998359,0.995522,0.995521,0.996639,49.678005


time: 19min 4s (started: 2026-05-17 00:13:32 +00:00)


time: 19min 4s (started: 2026-05-17 00:13:32 +00:00)


time: 19min 4s (started: 2026-05-17 00:13:32 +00:00)


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# GBM
name = "GBM"

# Base model
model = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=10,
    max_iter=5,
    random_state=42
)

# Hyperparameter tuning
param_dist = {
    'learning_rate': [0.1, 0.05],
    'max_depth': [5, 10],
    'max_iter': [100, 150]
}


# Train best model
best_model = hyper_parameter_tune(
    model,
    param_dist,
    X_train_scaled,
    y_train_sm
)

GBM = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
GBM

Best parameters: {'max_iter': 100, 'max_depth': 10, 'learning_rate': 0.1}
Best CV accuracy: 0.9987

Train Model: GBM



,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,GBM,1,0.998921,0.998922,0.998921,0.998921,0.999974,0.998562,0.998561,0.998918,41.577456
1,GBM,2,0.999063,0.999064,0.999063,0.999063,0.999981,0.998752,0.998751,0.999062,42.234813
2,GBM,3,0.999043,0.999044,0.999043,0.999043,0.999878,0.998725,0.998724,0.999050,41.587615
3,GBM,4,0.997923,0.997924,0.997923,0.997923,0.999466,0.997232,0.997231,0.997931,34.277656
4,GBM,5,0.998371,0.998372,0.998371,0.998371,0.999853,0.997829,0.997828,0.998364,31.461657
5,GBM,6,0.998493,0.998494,0.998493,0.998493,0.999920,0.997992,0.997991,0.998487,34.262038
6,GBM,7,0.998127,0.998128,0.998127,0.998127,0.999531,0.997503,0.997503,0.998134,36.803792
7,GBM,8,0.998432,0.998433,0.998432,0.998432,0.999978,0.997910,0.997910,0.998421,28.777801
8,GBM,9,0.998025,0.998026,0.998025,0.998025,0.999577,0.997367,0.997367,0.998029,30.314657
9,GBM,10,0.999287,0.999288,0.999287,0.999287,0.999992,0.999050,0.999050,0.999288,37.751900


time: 23min 22s (started: 2026-05-17 00:32:36 +00:00)


time: 23min 22s (started: 2026-05-17 00:32:36 +00:00)


time: 23min 22s (started: 2026-05-17 00:32:36 +00:00)


In [ ]:
# Catboost

name ="CatBoost"

# Model
model = CatBoostClassifier(
    iterations=100,
    depth=3,
    loss_function='MultiClass',
    random_strength=1,
    verbose=0
)

param_dist = {
    'l2_leaf_reg': [1, 3, 5, 7],
    'bagging_temperature': [0, 1, 5]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)

CatBoost = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
CatBoost

Best parameters: {'l2_leaf_reg': 1, 'bagging_temperature': 0}
Best CV accuracy: 0.8913

Train Model: CatBoost



,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,CatBoost,1,0.894214,0.895906,0.894214,0.894541,0.983491,0.859300,0.858946,0.894157,26.333577
1,CatBoost,2,0.895191,0.897635,0.895191,0.895605,0.983459,0.860811,0.860251,0.895146,26.242343
2,CatBoost,3,0.889712,0.891767,0.889712,0.890132,0.982759,0.853351,0.852941,0.889770,25.633188
3,CatBoost,4,0.894476,0.896741,0.894476,0.894926,0.982927,0.859768,0.859300,0.894578,25.184884
4,CatBoost,5,0.889957,0.891974,0.889957,0.890311,0.982554,0.853706,0.853262,0.889850,26.355227
5,CatBoost,6,0.892644,0.895010,0.892644,0.893076,0.982627,0.857376,0.856866,0.892572,26.255755
6,CatBoost,7,0.890303,0.892195,0.890303,0.890704,0.982098,0.854115,0.853746,0.890373,26.327680
7,CatBoost,8,0.891708,0.893649,0.891708,0.892103,0.982683,0.855998,0.855609,0.891615,26.428942
8,CatBoost,9,0.891239,0.893535,0.891239,0.891691,0.982765,0.855464,0.854991,0.891316,25.251043
9,CatBoost,10,0.890079,0.891985,0.890079,0.890479,0.982467,0.853813,0.853436,0.890128,25.450094


time: 14min 46s (started: 2026-05-17 00:55:59 +00:00)


In [ ]:
from sklearn.svm import SVC

# Model
model = SVC()

name="SVM"

# Param distribution
param_dist = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]
}

# Hyperparameter tuning
best_model = hyper_parameter_tune(
    model,
    param_dist,
    X_train_scaled,
    y_train_sm
)

SVM = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
SVM

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,SVM,1,0.999446,0.999447,0.999446,0.999446,NaN,0.999262,0.999261,0.999440,4.793383
1,SVM,2,0.999723,0.999723,0.999723,0.999723,NaN,0.999631,0.999631,0.999728,5.877714
2,SVM,3,0.999908,0.999908,0.999908,0.999908,NaN,0.999877,0.999877,0.999905,4.878195
3,SVM,4,0.999815,0.999815,0.999815,0.999815,NaN,0.999754,0.999754,0.999812,5.302073
4,SVM,5,0.999723,0.999723,0.999723,0.999723,NaN,0.999631,0.999631,0.999715,5.452982
5,SVM,6,0.999815,0.999815,0.999815,0.999815,NaN,0.999754,0.999754,0.999813,5.040118
6,SVM,7,0.999723,0.999723,0.999723,0.999723,NaN,0.999631,0.999631,0.999728,6.137564
7,SVM,8,0.999723,0.999723,0.999723,0.999723,NaN,0.999631,0.999631,0.999723,4.628765
8,SVM,9,0.999723,0.999723,0.999723,0.999723,NaN,0.999631,0.999631,0.999728,5.988212
9,SVM,10,0.999631,0.999631,0.999631,0.999631,NaN,0.999508,0.999508,0.999644,4.787380


time: 50.8 ms (started: 2026-05-19 08:40:21 +00:00)


In [ ]:
from sklearn.linear_model import LogisticRegression

name ="Logistic Regression" # change as per model

# Model
model = LogisticRegression(random_state=42, max_iter=1000, solver="liblinear", C = 1.0) # change as per model

# Parameters, change as per model
param_dist = {
        'C': [0.1, 0.6, 0.8, 1],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
}

best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)

Logistic_Regression = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
Logistic_Regression

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,Logistic Regression,1,0.951068,0.951375,0.951068,0.951142,0.994168,0.934807,0.934756,0.951076,34.057681
1,Logistic Regression,2,0.950544,0.950841,0.950544,0.950609,0.993956,0.934112,0.934057,0.950460,33.639572
2,Logistic Regression,3,0.952219,0.952552,0.952219,0.952278,0.994325,0.936363,0.936293,0.952324,35.779835
3,Logistic Regression,4,0.949027,0.949364,0.949027,0.949126,0.994382,0.932078,0.932032,0.949005,32.728733
4,Logistic Regression,5,0.953056,0.953313,0.953056,0.953103,0.993910,0.937456,0.937403,0.953084,31.209545
5,Logistic Regression,6,0.951225,0.951545,0.951225,0.951303,0.994150,0.935006,0.934955,0.951198,35.570221
6,Logistic Regression,7,0.949129,0.949394,0.949129,0.949185,0.994228,0.932222,0.932173,0.949137,30.048933
7,Logistic Regression,8,0.948553,0.948947,0.948553,0.948650,0.993576,0.931460,0.931395,0.948537,32.932031
8,Logistic Regression,9,0.949914,0.950097,0.949914,0.949964,0.994150,0.933242,0.933215,0.949831,33.075392
9,Logistic Regression,10,0.950228,0.950519,0.950228,0.950308,0.994160,0.933679,0.933637,0.950284,34.923320


time: 36.3 ms (started: 2026-05-19 08:40:39 +00:00)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

name = "Random Forest"

# Model
model = RandomForestClassifier(n_jobs=-1,
    random_state=42
)

# Parameters
param_dist = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)

Random_Forest = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
Random_Forest

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,Random Forest,1,0.988558,0.988653,0.988558,0.988562,0.999690,0.984773,0.984744,0.988541,240.456005
1,Random Forest,2,0.988192,0.988303,0.988192,0.988200,0.999683,0.984287,0.984256,0.988184,239.989493
2,Random Forest,3,0.988029,0.988139,0.988029,0.988038,0.999688,0.984068,0.984038,0.988014,237.088522
3,Random Forest,4,0.987662,0.987751,0.987662,0.987667,0.999691,0.983576,0.983549,0.987669,237.929663
4,Random Forest,5,0.986726,0.986859,0.986726,0.986735,0.999644,0.982338,0.982300,0.986686,241.613954
5,Random Forest,6,0.987255,0.987360,0.987255,0.987262,0.999629,0.983037,0.983007,0.987257,243.219578
6,Random Forest,7,0.987621,0.987739,0.987621,0.987631,0.999607,0.983528,0.983495,0.987645,240.266899
7,Random Forest,8,0.987540,0.987664,0.987540,0.987547,0.999623,0.983423,0.983386,0.987559,240.423835
8,Random Forest,9,0.987886,0.988016,0.987886,0.987896,0.999692,0.983885,0.983848,0.987904,237.978543
9,Random Forest,10,0.988354,0.988460,0.988354,0.988361,0.999706,0.984503,0.984472,0.988361,240.936251


time: 37.4 ms (started: 2026-05-19 08:40:50 +00:00)


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

name="BaggingClassifier"

model = BaggingClassifier(
    estimator=my_base_model,
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)
param_dist = {
    "n_estimators": [100],
    "max_samples": [0.75, 1.0],
    "max_features": [0.75, 1.0]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

Bagging = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
Bagging

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,BaggingClassifier,1,0.899446,0.900222,0.899446,0.899409,0.985994,0.866206,0.865922,0.899375,99.692653
1,BaggingClassifier,2,0.902541,0.903420,0.902541,0.902607,0.986109,0.870304,0.870051,0.902493,98.290943
2,BaggingClassifier,3,0.902417,0.903125,0.902417,0.902430,0.986282,0.870118,0.869888,0.902534,100.168385
3,BaggingClassifier,4,0.902233,0.903120,0.902233,0.902245,0.985781,0.869940,0.869648,0.902387,101.923950
4,BaggingClassifier,5,0.901480,0.902305,0.901480,0.901499,0.985697,0.868896,0.868628,0.901351,98.396888
5,BaggingClassifier,6,0.898243,0.899349,0.898243,0.898327,0.985343,0.864638,0.864321,0.898060,96.950112
6,BaggingClassifier,7,0.900930,0.901797,0.900930,0.900949,0.985696,0.868193,0.867915,0.901010,99.043154
7,BaggingClassifier,8,0.902905,0.903585,0.902905,0.902913,0.985959,0.870755,0.870532,0.902759,105.153057
8,BaggingClassifier,9,0.900035,0.900972,0.900035,0.900101,0.985556,0.866991,0.866719,0.900147,108.048750
9,BaggingClassifier,10,0.901399,0.902180,0.901399,0.901496,0.985789,0.868730,0.868532,0.901472,108.764375


time: 35.4 ms (started: 2026-05-19 08:41:02 +00:00)


In [ ]:
from lightgbm import LGBMClassifier

name = "LightGBM" # change as per model

# Model
model = LGBMClassifier(n_jobs=2, verbose=-1) # change as per model

# Parameters, change as per model
param_dist = {
        'n_estimators': [21, 51, 101],
        'learning_rate': [0.01, 0.05, 0.08],
        'num_leaves': [30, 50, 70],
        'max_depth': [-1, 5, 10],
        'subsample': [0.2, 0.5, 0.8],
        'colsample_bytree': [0.2, 0.5, 0.8]
    }

best_model = hyper_parameter_tune(model, param_dist, X_train_scaled, y_train_sm)
LightGBM = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
LightGBM

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,LightGBM,1,0.995847,0.995848,0.995847,0.995845,0.999934,0.994464,0.994462,0.995838,56.014437
1,LightGBM,2,0.996274,0.996275,0.996274,0.996273,0.999951,0.995034,0.995032,0.996269,24.201820
2,LightGBM,3,0.995724,0.995727,0.995724,0.995723,0.999945,0.994301,0.994299,0.995733,24.415209
3,LightGBM,4,0.995887,0.995889,0.995887,0.995886,0.999926,0.994518,0.994516,0.995897,27.952126
4,LightGBM,5,0.995684,0.995685,0.995684,0.995682,0.999934,0.994246,0.994245,0.995669,24.246713
5,LightGBM,6,0.995602,0.995603,0.995602,0.995600,0.999934,0.994138,0.994136,0.995595,25.331151
6,LightGBM,7,0.995582,0.995586,0.995582,0.995581,0.999938,0.994112,0.994109,0.995596,20.548093
7,LightGBM,8,0.995928,0.995931,0.995928,0.995927,0.999944,0.994572,0.994571,0.995913,21.111623
8,LightGBM,9,0.996356,0.996360,0.996356,0.996355,0.999956,0.995143,0.995141,0.996363,24.533791
9,LightGBM,10,0.996274,0.996276,0.996274,0.996273,0.999961,0.995034,0.995032,0.996275,23.833752


time: 37.1 ms (started: 2026-05-19 08:41:12 +00:00)


In [ ]:
name="KNN"

from sklearn.neighbors import KNeighborsClassifier

# Model
model = KNeighborsClassifier(n_jobs=-1);

# Param distribution
param_dist = {
    'n_neighbors': [3, 5, 7],     # keep small range
    'weights': ['uniform'],          # distance = slower
    'p': [2],                        # Euclidean only (faster)
    'algorithm': ['kd_tree']         # fastest in most cases
}

# Hyperparameter tuning
best_model = hyper_parameter_tune(
    model,
    param_dist,
    X_train_scaled,
    y_train_sm
)

KNN = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
KNN

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,KNN,1,0.998880,0.998882,0.998880,0.998880,0.999595,0.998508,0.998507,0.998876,5.776723
1,KNN,2,0.998921,0.998923,0.998921,0.998920,0.999592,0.998562,0.998561,0.998918,4.726142
2,KNN,3,0.998534,0.998537,0.998534,0.998533,0.999504,0.998047,0.998045,0.998547,5.922305
3,KNN,4,0.998819,0.998821,0.998819,0.998819,0.999558,0.998426,0.998426,0.998829,4.767427
4,KNN,5,0.998758,0.998760,0.998758,0.998757,0.999482,0.998345,0.998344,0.998751,4.947465
5,KNN,6,0.998758,0.998760,0.998758,0.998757,0.999453,0.998345,0.998344,0.998747,6.049948
6,KNN,7,0.998697,0.998699,0.998697,0.998696,0.999557,0.998264,0.998263,0.998703,4.816549
7,KNN,8,0.999125,0.999125,0.999125,0.999124,0.999726,0.998833,0.998833,0.999116,4.885449
8,KNN,9,0.999002,0.999004,0.999002,0.999002,0.999649,0.998670,0.998670,0.999008,4.726678
9,KNN,10,0.998962,0.998963,0.998962,0.998961,0.999651,0.998616,0.998616,0.998965,4.725839


time: 54 ms (started: 2026-05-19 08:41:20 +00:00)


In [ ]:
name = "StackingClassifier"

from sklearn.ensemble import StackingClassifier
#Model
model = StackingClassifier(
          cv=3,
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42, max_depth=2)),
        ],
        n_jobs=-1,
        verbose=0,
         )
param_dist = {
       'passthrough': [False],
        'final_estimator': [LogisticRegression(random_state=42, max_iter=100)]
}

# Hyperparameter tuning
best_model = hyper_parameter_tune(
    model,
    param_dist,
    X_train_scaled,
    y_train_sm
)

Stacking = evaluate_models(name, best_model, X_train_scaled, y_train_sm)
Stacking

,Classifier,Fold,Accuracy,Precision,Recall,F1,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,StackingClassifier,1,0.929496,0.929710,0.929496,0.929570,0.989126,0.906017,0.905995,0.929509,40.924548
1,StackingClassifier,2,0.927929,0.928343,0.927929,0.928033,0.989482,0.903972,0.903905,0.927937,43.022943
2,StackingClassifier,3,0.932386,0.932586,0.932386,0.932444,0.989974,0.909871,0.909844,0.932402,42.043369
3,StackingClassifier,4,0.927663,0.927886,0.927663,0.927733,0.989463,0.903575,0.903548,0.927684,40.745109
4,StackingClassifier,5,0.927479,0.927811,0.927479,0.927574,0.988757,0.903350,0.903303,0.927501,40.738126
5,StackingClassifier,6,0.929821,0.930178,0.929821,0.929926,0.988898,0.906477,0.906429,0.929814,41.119242
6,StackingClassifier,7,0.927174,0.927539,0.927174,0.927290,0.989001,0.902941,0.902898,0.927156,41.049913
7,StackingClassifier,8,0.929780,0.930177,0.929780,0.929893,0.989299,0.906428,0.906372,0.929743,40.947556
8,StackingClassifier,9,0.927398,0.927639,0.927398,0.927468,0.989101,0.903229,0.903196,0.927375,41.322491
9,StackingClassifier,10,0.929637,0.929880,0.929637,0.929692,0.989556,0.906224,0.906180,0.929608,40.824412


time: 66.7 ms (started: 2026-05-19 08:41:31 +00:00)


In [ ]:
import pandas as pd

# Create one Excel file with separate sheets
with pd.ExcelWriter('/content/all_models.xlsx') as writer:

    CatBoost.to_excel(writer, sheet_name='CatBoost', index=False)

    Logistic_Regression.to_excel(writer, sheet_name='Logistic_Regression', index=False)

    GBM.to_excel(writer, sheet_name='GBM', index=False)

    MLP.to_excel(writer, sheet_name='MLP', index=False)

    XGBoost.to_excel(writer, sheet_name='XGBoost', index=False)

    DecisionTree.to_excel(writer, sheet_name='DecisionTree', index=False)

    Random_Forest.to_excel(writer, sheet_name='Random_Forest', index=False)

    KNN.to_excel(writer, sheet_name='KNN', index=False)

    Bagging.to_excel(writer, sheet_name='Bagging', index=False)

    Stacking.to_excel(writer, sheet_name='Stacking', index=False)

    LightGBM.to_excel(writer, sheet_name='LightGBM', index=False)

    SVM.to_excel(writer, sheet_name='SVM', index=False)

print("All DataFrames saved into one Excel file with separate sheets!")

All DataFrames saved into one Excel file with separate sheets!
time: 51 ms (started: 2026-05-19 08:29:05 +00:00)


In [ ]:
!pip install matplotlib-venn
!apt-get -qq install -y libfluidsynth1

# https://pypi.python.org/pypi/libarchive
# !apt-get -qq install -y libarchive-dev && pip install -U libarchive
# import libarchive

# https://pypi.python.org/pypi/pydot
!apt-get -qq install -y graphviz && pip install pydot
import pydot

!pip install cartopy
import cartopy

E: Package 'libfluidsynth1' has no installation candidate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 117.4 MB/s eta 0:00:00
time: 21.7 s (started: 2026-05-19 08:32:45 +00:00)


In [ ]:
import glob
import os

!pip install xlsxwriter

folder_path = "/content/drive/MyDrive/"
csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
# csv_files.remove('/content/bank-additional-full.csv')
metrics = [
    "Accuracy", "Precision", "Recall",
    "F1", "AUC", "MCC", "Kappa",
    "Balanced Accuracy"
]

metric_tables = {metric: pd.DataFrame() for metric in metrics}

for file in csv_files:
    df = pd.read_csv(file)

    name = df.get("Classifier").iloc[0]
    # name = os.path.splitext(os.path.basename(file))[0]
    for metric in metrics:
        temp = df[["Fold", metric]].copy()
        temp = temp.set_index("Fold")
        metric_tables[metric][name] = temp[metric]

output_file = "all_model.xlsx"

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    for metric, table in metric_tables.items():
        table = table.sort_index()
        sheet_name = metric[:31]
        table.to_excel(writer, sheet_name=sheet_name)

print("Excel file created:", output_file)

Excel file created: all_model.xlsx
time: 4.29 s (started: 2026-05-19 08:36:30 +00:00)


In [ ]:
# wilcoxon
import pandas as pd
from scipy.stats import wilcoxon
import itertools

file_path = "all_models.xlsx"

xls = pd.ExcelFile(file_path)
sheets = xls.sheet_names

results = []

for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)

    df = df.set_index(df.columns[0])
    models = df.columns

    for model1, model2 in itertools.combinations(models, 2):
        x = df[model1]
        y = df[model2]


        stat, p = wilcoxon(x, y, nan_policy='omit')

        results.append({
            "Metric": sheet,
            "Reference Model": model1,
            "Compared Model": model2,
            "Wilcoxon Statistic": stat,
            "p-value": p,
            "Significant (p < 0.05)": p < 0.05
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Save
output_file = "Wilcoxon_Signed_Rank_Test_results.xlsx"
results_df.to_excel(output_file, index=False)

print("Saved :", output_file)
results_df

/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: Runtim

Saved : Wilcoxon_Signed_Rank_Test_results.xlsx


,Metric,Reference Model,Compared Model,Wilcoxon Statistic,p-value,Significant (p < 0.05)
0,CatBoost,Fold,Accuracy,0.0,0.001953,True
1,CatBoost,Fold,Precision,0.0,0.001953,True
2,CatBoost,Fold,Recall,0.0,0.001953,True
3,CatBoost,Fold,F1,0.0,0.001953,True
4,CatBoost,Fold,AUC,0.0,0.001953,True
...,...,...,...,...,...,...
535,SVM,MCC,Balanced Accuracy,0.0,0.001953,True
536,SVM,MCC,Training Time (s),0.0,0.001953,True
537,SVM,Kappa,Balanced Accuracy,0.0,0.001953,True
538,SVM,Kappa,Training Time (s),0.0,0.001953,True


time: 52.6 s (started: 2026-05-19 08:36:43 +00:00)


In [ ]:
results_df.to_csv("Wilcoxon_Signed_Rank_Test_results.csv", index=False)

from google.colab import files
files.download("Wilcoxon_Signed_Rank_Test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

time: 60.3 ms (started: 2026-05-19 08:39:36 +00:00)
